# 20_02 — Logistic Regression on ResNet50 Embeddings (Cached)

## Goal
Train and benchmark a **Logistic Regression** classifier on **cached ResNet50 embeddings** extracted in `20_01_extract_embeddings_resnet50.ipynb`.

This notebook:
- **Does not load images** (uses `.npy` embeddings only) → fast
- **Is re-runnable** (loads cached embeddings every time)
- **Is parallel-safe** (unique `RUN_ID`, no artifact collisions)
- Produces:
  - `models/ml_deep_features/resnet50_embeddings/run_YYYYMMDD_HHMMSS/classifier.pkl`
  - `config.json`, `metrics.json`
  - `reports/metrics/resnet50_lr_metrics.json`
- Logs experiment to **MLflow** (`mlruns/`)

## Required cache
Embeddings must exist under:
`data/processed/embeddings/split_v1/encoder_resnet50/`

In [ ]:
# Cell 1 — Imports

import json
import pickle
from datetime import datetime
from pathlib import Path

import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

import mlflow

In [ ]:
# Cell 2 — Project root + paths + run metadata (robust)

NOTEBOOK_DIR = Path.cwd().expanduser().resolve()

def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "pyproject.toml", "requirements.txt"]

    for p in [start] + list(start.parents):
        has_all = all((p / m).exists() for m in markers_all)
        has_any = any((p / m).exists() for m in markers_any)
        if has_all and has_any:
            return p

    for p in [start] + list(start.parents):
        if all((p / m).exists() for m in markers_all):
            return p

    raise RuntimeError(f"Could not locate project root from: {start}")

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
CONFIGS_DIR = PROJECT_ROOT / "configs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_ID = "split_v1"

# Embedding cache directory (from 20_01)
EMB_DIR = DATA_DIR / "processed" / "embeddings" / SPLIT_ID / "encoder_resnet50"

MODEL_NAME = "resnet50_embeddings"
FEATURE_TYPE = "resnet50_embeddings"
CLASSIFIER_NAME = "logreg"
TRANSFORM_ID = "transforms_v1_eval_resize256_centercrop224_imagenetnorm"

RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")

RUN_DIR = MODELS_DIR / "ml_deep_features" / MODEL_NAME / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)

REPORT_METRICS_PATH = REPORTS_METRICS_DIR / "resnet50_lr_metrics.json"
REPORT_METRICS_PATH_VARIANT = REPORTS_METRICS_DIR / f"resnet50_lr_{RUN_ID}.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EMB_DIR     :", EMB_DIR)
print("RUN_DIR     :", RUN_DIR)

In [ ]:
# Cell 3 — MLflow setup

TRACKING_DIR = PROJECT_ROOT / "mlruns"
TRACKING_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

print("MLflow tracking URI:", TRACKING_DIR.as_uri())

In [ ]:
# Cell 4 — Embedding cache check (hard gate)

paths = {
    "train": EMB_DIR / "train.npy",
    "val": EMB_DIR / "val.npy",
    "test": EMB_DIR / "test.npy",
    "labels_train": EMB_DIR / "labels_train.npy",
    "labels_val": EMB_DIR / "labels_val.npy",
    "labels_test": EMB_DIR / "labels_test.npy",
    "meta": EMB_DIR / "meta.json",
}

missing = [k for k, p in paths.items() if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Embedding cache is missing required files:\n"
        + "\n".join([f" - {k}: {paths[k]}" for k in missing])
        + "\n\nRun notebooks/20_ml_deep_features_fixed_encoder/20_01_extract_embeddings_resnet50.ipynb first."
    )

print("[OK] Found embedding cache files.")

In [ ]:
# Cell 5 — Load embeddings (memory-aware)

X_train = np.load(paths["train"], mmap_mode="r")
X_val = np.load(paths["val"], mmap_mode="r")
X_test = np.load(paths["test"], mmap_mode="r")

y_train = np.load(paths["labels_train"])
y_val = np.load(paths["labels_val"])
y_test = np.load(paths["labels_test"])

EMBEDDING_DIM = X_train.shape[1]

print("Shapes:")
print(" - X_train:", X_train.shape, "y_train:", y_train.shape)
print(" - X_val  :", X_val.shape, "y_val  :", y_val.shape)
print(" - X_test :", X_test.shape, "y_test :", y_test.shape)
print("Embedding dim:", EMBEDDING_DIM)

In [ ]:
# Cell 6 — Load class mapping (for readable classification report)

CLASSES_JSON = DATA_DIR / "splits" / SPLIT_ID / "classes.json"
with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    class_to_idx = json.load(f)

idx_to_class = {v: k for k, v in class_to_idx.items()}
target_names = [idx_to_class[i] for i in sorted(idx_to_class.keys())]

print("Classes:", target_names)

In [ ]:
# Cell 7 — Sanity checks

assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]

assert np.isfinite(np.asarray(X_train[:2000])).all()
assert set(np.unique(y_train)).issubset(set(class_to_idx.values()))

print("[OK] Sanity checks passed.")

In [ ]:
# Cell 8 — Train Logistic Regression (scalable defaults)

# Important: Different sklearn versions vary; keep params conservative.
# 'saga' scales well; omit 'multi_class' for compatibility.
LOGREG_PARAMS = {
    "solver": "saga",
    "C": 2.0,
    "max_iter": 1000,
    "random_state": 42,
    "n_jobs": -1,
}

# Scaling is optional for embeddings; keep it lightweight and memmap-friendly.
# with_mean=False avoids densifying/centering overhead.
model = Pipeline([
    ("scaler", StandardScaler(with_mean=False, with_std=True)),
    ("clf", LogisticRegression(**LOGREG_PARAMS)),
])

model.fit(X_train, y_train)
print("[OK] Training complete.")

In [ ]:
# Cell 9 — Evaluation

def evaluate(model, X, y):
    preds = model.predict(X)
    metrics = {
        "accuracy": float(accuracy_score(y, preds)),
        "f1_macro": float(f1_score(y, preds, average="macro")),
        "precision_macro": float(precision_score(y, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y, preds, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(y, preds).tolist(),
        "classification_report": classification_report(
            y, preds, target_names=target_names, zero_division=0, output_dict=True
        ),
    }
    return metrics

val_metrics = evaluate(model, X_val, y_val)
test_metrics = evaluate(model, X_test, y_test)

print("VAL  accuracy:", val_metrics["accuracy"], "VAL  f1_macro:", val_metrics["f1_macro"])
print("TEST accuracy:", test_metrics["accuracy"], "TEST f1_macro:", test_metrics["f1_macro"])

In [ ]:
# Cell 10 — Save artifacts

# Per plan, deep-feature model uses classifier.pkl
classifier_path = RUN_DIR / "classifier.pkl"
config_path = RUN_DIR / "config.json"
metrics_path = RUN_DIR / "metrics.json"

# include meta.json reference (and optionally its content)
with open(paths["meta"], "r", encoding="utf-8") as f:
    emb_meta = json.load(f)

config = {
    "model_name": MODEL_NAME,
    "feature_type": FEATURE_TYPE,
    "classifier": CLASSIFIER_NAME,
    "split_id": SPLIT_ID,
    "transform_id": TRANSFORM_ID,
    "run_id": RUN_ID,
    "embedding_dir": str(EMB_DIR),
    "embedding_dim": int(EMBEDDING_DIM),
    "logreg_params": LOGREG_PARAMS,
    "embedding_meta": emb_meta,
}

metrics_payload = {"val": val_metrics, "test": test_metrics}

with open(classifier_path, "wb") as f:
    pickle.dump(model, f)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

# Plan-stable name (overwritten if rerun; acceptable per plan)
with open(REPORT_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

# Variant-safe name (never overwritten)
with open(REPORT_METRICS_PATH_VARIANT, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print("Saved:")
print(" -", classifier_path)
print(" -", config_path)
print(" -", metrics_path)
print(" -", REPORT_METRICS_PATH)
print(" -", REPORT_METRICS_PATH_VARIANT)

In [ ]:
# Cell 11 — MLflow logging

with mlflow.start_run(run_name=f"{MODEL_NAME}_{CLASSIFIER_NAME}_{RUN_ID}"):
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("feature_type", FEATURE_TYPE)
    mlflow.log_param("classifier", CLASSIFIER_NAME)
    mlflow.log_param("split_id", SPLIT_ID)
    mlflow.log_param("transform_id", TRANSFORM_ID)
    mlflow.log_param("run_id", RUN_ID)

    mlflow.log_param("embedding_dir", str(EMB_DIR))
    mlflow.log_param("embedding_dim", int(EMBEDDING_DIM))

    for k, v in LOGREG_PARAMS.items():
        mlflow.log_param(f"logreg_{k}", v)

    mlflow.log_metric("val_accuracy", val_metrics["accuracy"])
    mlflow.log_metric("val_f1_macro", val_metrics["f1_macro"])
    mlflow.log_metric("test_accuracy", test_metrics["accuracy"])
    mlflow.log_metric("test_f1_macro", test_metrics["f1_macro"])

    mlflow.log_artifact(str(classifier_path))
    mlflow.log_artifact(str(config_path))
    mlflow.log_artifact(str(metrics_path))

print("[OK] MLflow run logged.")

In [ ]:
# Cell 12 — Verification: reload + inference sanity check

with open(classifier_path, "rb") as f:
    loaded_model = pickle.load(f)

sample_X = np.asarray(X_test[:8])
sample_preds = loaded_model.predict(sample_X)
print("Sample preds (idx):", sample_preds.tolist())
print("Sample preds (label):", [target_names[int(i)] for i in sample_preds])